# Exercise C: NAIRR Open Datasets

**Estimated time:** ~30 minutes

NAIRR provides free datasets through its [Pilot Resources page](https://nairrpilot.org/pilotresources) and through [HuggingFace](https://huggingface.co/datasets). Researchers can browse and load datasets directly into notebooks — no special authentication required for most resources.

This exercise demonstrates:
- How to browse NAIRR's dataset offerings
- Loading a substantial real-world dataset (120K documents) with one line of code
- Text vectorization, classifier training, and evaluation
- **Timing differences between your local machine and Jetstream2**

In [ ]:
!pip install datasets pandas numpy matplotlib scikit-learn --quiet

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


class timed:
    """Context manager for timing code blocks."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] {self.elapsed:.2f} seconds")

## Browsing NAIRR Datasets

Before loading data programmatically, it is worth browsing what is available:

- **NAIRR Pilot Resources:** [nairrpilot.org/pilotresources](https://nairrpilot.org/pilotresources) — curated datasets from NAIRR partners (health, climate, materials science, social science)
- **HuggingFace Datasets:** [huggingface.co/datasets](https://huggingface.co/datasets) — 200,000+ datasets, freely accessible, one-line loading
- **NVIDIA on HuggingFace:** [huggingface.co/nvidia](https://huggingface.co/nvidia) — 180+ datasets with permissive licenses

For this exercise, we use the **AG News** dataset from HuggingFace — 120,000 news articles classified into 4 categories. This is large enough to show meaningful timing differences between local and Jetstream2.

In [ ]:
from datasets import load_dataset

with timed("Download AG News dataset"):
    dataset = load_dataset("ag_news", split="train")

print(f"Loaded {len(dataset):,} samples")
print(f"Features: {dataset.column_names}")
print(f"Labels: {dataset.features['label'].names}")

In [ ]:
with timed("Convert to DataFrame"):
    df = dataset.to_pandas()

print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024 ** 2:.1f} MB\n")

# Map label indices to names
label_names = dataset.features["label"].names
df["label_name"] = df["label"].map(lambda x: label_names[x])

print("Class distribution:")
print(df["label_name"].value_counts())

In [ ]:
# Text length statistics
df["text_length"] = df["text"].str.len()
print("Text length statistics:")
print(df["text_length"].describe().to_string())

# Show a sample from each category
print("\n" + "=" * 80)
for label in label_names:
    sample = df[df["label_name"] == label].iloc[0]
    print(f"\n[{label}]")
    print(f"  {sample['text'][:200]}...")

## Step 1: TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) converts text into numerical feature vectors. With 120,000 documents and 10,000 features, this is a compute-intensive operation — and the kind of preprocessing that slows down on laptops with limited memory.

In [ ]:
with timed("TF-IDF vectorization (120K documents, 10K features)"):
    tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
    X = tfidf.fit_transform(df["text"])

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Matrix density: {X.nnz / (X.shape[0] * X.shape[1]):.4%}")
print(f"Memory (sparse): {X.data.nbytes / 1024 ** 2:.1f} MB")

## Step 2: Train a Text Classifier

We split the data 80/20 and train a Logistic Regression classifier. Logistic Regression is a strong baseline for text classification and trains quickly, making it a practical choice for research iteration.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

print(f"Training set: {X_train.shape[0]:,} documents")
print(f"Test set:     {X_test.shape[0]:,} documents\n")

with timed("Train LogisticRegression on 96K documents"):
    clf = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
    clf.fit(X_train, y_train)

with timed("Predict on 24K test documents"):
    y_pred = clf.predict(X_test)

In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("AG News Classification — Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())

print("Top 10 most predictive words per category:\n")
for i, label in enumerate(label_names):
    top_indices = clf.coef_[i].argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    print(f"  {label}: {', '.join(top_words)}")

## Timing Comparison

| Operation                          | Local (your laptop) | Jetstream2 |
|------------------------------------|--------------------:|-----------:|
| Download AG News dataset           |          ___ sec    |   ___ sec  |
| Convert to DataFrame               |          ___ sec    |   ___ sec  |
| TF-IDF vectorization (120K docs)   |          ___ sec    |   ___ sec  |
| Train LogisticRegression (96K)     |          ___ sec    |   ___ sec  |
| Predict (24K)                      |          ___ sec    |   ___ sec  |
| **Total pipeline**                 |      **___ sec**    |**___ sec** |

**Fill in both columns.** The TF-IDF vectorization step typically shows the largest difference because it is both CPU and memory intensive.

## Try Another Dataset

HuggingFace has 200,000+ datasets across every domain. Try loading one relevant to your research — the pattern is always the same: one line to load, convert to DataFrame, explore.

In [ ]:
# Uncomment and modify to try a different dataset:

# from datasets import load_dataset

# Example datasets to try:
# ds = load_dataset("rotten_tomatoes", split="train")           # 8.5K movie reviews
# ds = load_dataset("imdb", split="train")                      # 25K movie reviews
# ds = load_dataset("tweet_eval", "sentiment", split="train")   # Tweet sentiment
# ds = load_dataset("scikit-learn/iris", split="train")          # Classic Iris (tiny)

# ds = load_dataset("YOUR_CHOSEN_DATASET", split="train[:5000]")  # First 5000 samples
# df_new = ds.to_pandas()
# print(f"Shape: {df_new.shape}")
# print(f"Columns: {df_new.columns.tolist()}")
# df_new.head()

## Research Implications

- With one line of code, you loaded 120,000 labeled documents and built a text classifier
- No downloading CSVs, no cleaning headers, no dealing with encoding issues
- The NAIRR ecosystem provides both the **data** and the **compute** to work with it
- As your datasets grow larger and your models grow more complex, NAIRR's compute advantage becomes increasingly important
- The same workflow runs on your laptop, on Jetstream2, or on any other NAIRR system — the only thing that changes is the speed